# 02 — Data Preprocessing | IDX Exchange
**Week 3 deliverable**

**Goal:** Generate cleaned California single-family property rows, document every filtering decision, and create chronological train/validation/test partitions *without using future prices to fit preprocessing*. The AVM manual requires a **last-complete-month test**, **previous-month validation**, training-derived price outlier cutoffs, and no list-price features.

The cleaned CSV is written locally under `outputs/` (not included in GitHub). The imputer, scaling and categorical encoding are fitted **later, on training data only** as part of the model pipeline, not on the combined CSV.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RAW_DIR = ROOT / 'data' / 'raw'
OUTPUT_DIR = ROOT / 'outputs'
from src import avm
print('Project:', ROOT)
print('Input:', RAW_DIR)


### 1. Parse raw MLS sources and count record removal

In [ ]:
raw, filenames = avm.load_raw(RAW_DIR)
clean, audit = avm.clean_properties(raw)
import pandas as pd
counts = pd.Series(audit, name="remaining_rows").to_frame()
counts["removed_since_raw"] = audit["raw"] - counts["remaining_rows"]
display(counts)

### 2. Save the cleaned dataset locally, but **never commit it**

In [ ]:
clean_path = avm.write_clean_local_csv(clean, OUTPUT_DIR)
print("Cleaned CSV saved:", clean_path)
print("Records:", len(clean))

### 3. Chronological partition and missing/incomplete month handling
`AS_OF` lets the project reproduce the exact month split at the time of the meeting. Most recent *completed* month (among source data) is test; preceding month validation.

In [ ]:
AS_OF = "2026-10-12"   # update for a future report / data snapshot
splits, split_info = avm.chronological_split(clean, as_of=AS_OF)
print(split_info)

### 4. Freeze 0.5th / 99.5th target-price bounds from training only

In [ ]:
bounded_splits, limits = avm.price_outlier_filter(splits)
print(limits)
print("Before:", {k:len(v) for k,v in splits.items()})
print("After: ", {k:len(v) for k,v in bounded_splits.items()})

### 5. Identify predictive features and inspect missingness by month
Do not impute the target, do not train on `ListPrice`, `OriginalListPrice`, DOM, or any post-close field.

In [ ]:
num_cols, cat_cols = avm.choose_features(bounded_splits["train"])
print("Numeric predictors:", num_cols)
print("Categorical predictors:", cat_cols)
for name, part in bounded_splits.items():
    print("\n", name, len(part), "rows")
    display((part[num_cols + cat_cols].isna().mean()*100).round(1).rename("pct_missing").to_frame().head(20))

### Preprocessing decisions
- Missing *target* price is dropped; missing predictor values are **imputed later** with training-only median/mode.
- Numeric missingness indicators preserve information about missing fields.
- One-hot encoding handles the regional categories and unknown categories from the future month, with a cap on cardinality.
- The test month is set aside until the final model has been selected. No random split.
- Price outlier cutoffs are estimated on the training data alone, as requested by the supplied best-practices manual. This yields a *bounded-population* score and will require a separate unfiltered robustness report later.

**Outputs:** `outputs/california_single_family_cleaned.csv` and the documented pipeline artifacts when run from the CLI.
